# 03 -- Model Training

This notebook demonstrates the training methodology end to end: preprocessing
fit on TRAIN only, one imbalance strategy applied, and one model of each
family fit, using a fast subsample so the notebook runs in seconds.

**The authoritative, full-scale (700k-row train fold) result for every
model x imbalance-strategy combination is produced by `train.py` and saved
to `reports/metrics/model_comparison.csv` -- loaded and displayed at the
bottom of this notebook. Nothing in this notebook overrides those numbers;
this notebook exists to show the mechanism, not to re-derive the headline
results (that would just re-run train.py inside a notebook).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config

cfg = load_config(ROOT / "config.yaml")
pd.set_option("display.max_columns", 40)


In [2]:
from src.data_loader import load_and_split
from src.preprocessing import Preprocessor
from src import models, imbalance, evaluation

train_df, val_df, test_df = load_and_split(cfg)

# Demonstration subsample -- fast, illustrative only. The real comparison
# table below comes from the full 700k-row train fold via train.py.
demo_train = train_df.sample(n=60000, random_state=cfg.seed)
demo_val = val_df.sample(n=15000, random_state=cfg.seed)

pre = Preprocessor(cfg)
pre.fit(demo_train)
X_tr_tree, y_tr = pre.transform_tree(demo_train), pre.get_target(demo_train)
X_va_tree, y_va = pre.transform_tree(demo_val), pre.get_target(demo_val)
print(X_tr_tree.shape, X_va_tree.shape, y_tr.mean(), y_va.mean())

(60000, 52) (15000, 52) 0.011383333333333334 0.013533333333333333


## Train one LightGBM model with the `class_weight` (scale_pos_weight) strategy

In [3]:
X_res, y_res, kwargs = imbalance.apply_strategy(X_tr_tree, y_tr, "class_weight", cfg, cfg.seed)
model = models.train_lightgbm(X_res, y_res, X_va_tree, y_va, cfg, kwargs, cfg.seed)
p_va = models.predict_proba(model, X_va_tree, "lightgbm")
evaluation.evaluate_scores(y_va, p_va, cfg.evaluation.target_fpr_for_tpr, label="demo LightGBM / class_weight (60k-row subsample)")
print("\nNOTE: this is a small-subsample demo run for illustration; see the full-scale table below for the real reported numbers.")


NOTE: this is a small-subsample demo run for illustration; see the full-scale table below for the real reported numbers.


## The full-scale ablation result (all 4 models x 5 imbalance strategies, real 700k-row train fold)

Produced by running `python train.py` from the project root.

In [4]:
import pandas as pd
comparison_path = ROOT / "reports" / "metrics" / "model_comparison.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path, index_col=0)
    display(comparison[["roc_auc", "pr_auc", "tpr_at_5pct_fpr", "train_rows", "train_seconds"]].round(4))
else:
    print("Run `python train.py` from the project root first to produce this table.")

,roc_auc,pr_auc,tpr_at_5pct_fpr,train_rows,train_seconds
model__strategy,,,,,
logistic_regression__none,0.8766,0.1431,0.5039,700000.0,3.32
logistic_regression__class_weight,0.8775,0.1409,0.5027,700000.0,7.13
logistic_regression__random_undersample,0.8769,0.1425,0.5027,84920.0,0.45
logistic_regression__smote,0.8756,0.1424,0.4949,761508.0,4.99
logistic_regression__smote_undersample,0.8763,0.1411,0.4967,200720.0,1.75
random_forest__none,0.8725,0.1367,0.4900,700000.0,271.68
random_forest__class_weight,0.8697,0.1222,0.4792,700000.0,180.36
random_forest__random_undersample,0.8788,0.1446,0.5033,84920.0,25.51
random_forest__smote,0.8751,0.1280,0.4785,149999.0,40.86
